# Kaggriculture RL v2.1 — animals, hired hands, learned selling, self-play

Everything in v2 (real-env training, Δbank-only reward, masked factorized PPO with GAE) plus:

| | |
|---|---|
| **v2 fixes integrated** | BC class-imbalance fix (weighted CE + 1 DAgger round + logit-prior calibration → greedy **$3,834**, beats the $3,586 expert); head-temperature calibration + entropy schedule + value warmup (no more PPO collapse); `final_moneys` expert-contamination fixed |
| **Animals** | coops/pastures (free to build), buy → shed → pickup → place → feed(wheat) → care → collect fertilizer → harvest eggs/milk/wool |
| **Hired hands** | HIRE market action (first hand/day costs $1); every unit (farmer + up to 4 hands) gets its own farmer-head action each turn from a **unit-perspective** encoding |
| **Learned selling** | SELL_\<product\> market actions; curriculum phase 1 = scripted auto-sell, phase 2 = learned timing |
| **League self-play** | frozen policy snapshots join the opponent pool; per-opponent score tracking |
| **Throughput** | optional multiprocessing runner (`mp_worker.py`, shipped alongside) — 3.2× at 32 envs × 4 workers |

**Key mechanics** (verified against the env source, all encoded in the masks): coops/pastures cost $0; bought animals land in the **shed** (pickup from a center tile, then PLACE while standing on the structure); FEED consumes wheat from the **acting unit's** inventory; hands are dismissed every night and all inventories drop to the shed, so animal feeding needs a daily wheat re-pickup; 2 consecutive unfed days → animal escapes.

## 0. Setup & config

Same discipline as v2: one canonical config. **`SMOKE = True`** runs everything tiny for cell-by-cell testing; **set `SMOKE = False` for the real run.**

In [ ]:
# On Colab, uncomment first:
!pip install -U kaggle-environments
# from google.colab import drive; drive.mount('/content/drive')

import os
import time
import copy
import math
import json
import inspect
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from kaggle_environments import make

# ---- the one switch for step-by-step testing -------------------------------
SMOKE = False   # <- set to False for the full training run
# -----------------------------------------------------------------------------

CONFIG = dict(
    # environment
    n_envs          = 4    if SMOKE else 16,
    episode_steps   = 96   if SMOKE else 720,
    starting_money  = 3000.0,
    max_hands       = 4,                      # policy cap; HIRE masked beyond this
    # ppo
    epochs          = 3    if SMOKE else 300,
    gamma           = 0.999,
    lam             = 0.95,
    clip_eps        = 0.2,
    ppo_epochs      = 2    if SMOKE else 4,
    mini_batch      = 1024 if SMOKE else 4096,
    lr              = 3e-4,
    ent_coef_start  = 0.01,                   # linear decay to ent_coef_end (PPO-collapse fix)
    ent_coef_end    = 0.005,
    value_warmup_epochs = 2,                  # value-head-only epochs at PPO start
    vf_coef         = 0.5,
    max_grad_norm   = 0.5,
    reward_scale    = 0.01,
    obs_dtype       = torch.float16,          # rollout storage dtype (memory)
    # behavior cloning (winner recipe from the v2 fix experiments)
    use_bc_init     = True,
    bc_episodes     = 1    if SMOKE else 2,   # expert episodes per env
    bc_epochs       = 2    if SMOKE else 4,
    bc_lr           = 1e-3,
    bc_dagger_rounds = 1,
    bc_w_buy        = 20.0,                   # market-head sample weights
    bc_w_noop       = 50.0,
    bc_w_dagger     = 4.0,
    bc_tau_farmer   = 1.5,                    # logit-prior calibration strengths
    bc_tau_market   = 0.3,
    # post-BC temperature calibration (entropy target for PPO start)
    target_entropy  = 0.4,                      # 0.8 overheats a GOOD init (destroys the warm start)
    # BC-KL anchor: keep the policy near the expert early, annealed to 0
    anchor_coef_start = 0.05,
    anchor_coef_end   = 0.0,
    anchor_epochs     = 15,
    greedy_check_every = 5,                     # in-training greedy eval (mode-collapse alarm)
    # curriculum: phase 1 scripted auto-sell, phase 2 learned selling + self-play
    phase2_epoch    = 3    if SMOKE else 60,  # learned selling + snapshots start here
    wheat_reserve   = 4,                      # never auto-sell the last N wheat (feed)
    snapshot_every  = 1    if SMOKE else 10,
    league_mix_p1   = {"starter": 0.6, "random": 0.4, "snapshot": 0.0},
    league_mix_p2   = {"starter": 0.3, "random": 0.2, "snapshot": 0.5},
    max_snapshots   = 8,
    # io
    log_every       = 1 if SMOKE else 5,
    ckpt_every      = 1 if SMOKE else 25,
)

WORKDIR = os.getcwd()   # on Colab: '/content/drive/MyDrive/kaggriculture_rl'
CKPT_DIR = os.path.join(WORKDIR, "checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)
print("device:", device, "| SMOKE:", SMOKE, "| workdir:", WORKDIR)

## 1. The v2.1 contract: constants, pricing, action tables

Verified against the installed env source (`kaggle_environments/envs/kaggriculture/kaggriculture.py`) — 75/75 legality checks passed in the contract test suite. **v2 indices are frozen** (farmer 0–14, market 0–7), so v2 knowledge carries over.

* **Farmer (27)**: v2's 15 + BUILD_COOP/PASTURE, PLACE_GOOSE/COW/SHEEP, FEED, CARE, COLLECT_FERTILIZER, PICKUP_WHEAT, PICKUP_GOOSE/COW/SHEEP.
* **Market (22)**: v2's 8 + BUY_WHEAT, BUY_ANIMAL×3, HIRE, SELL×9 (one per product).
* `market_price` is an exact replica of the env's price function — used to mask buys against the *post-buy* quote, exactly like the env.

In [ ]:
# --------------------------------------------------------------------------
# Game constants (mirrored from the installed env source; do not edit base.py)
# --------------------------------------------------------------------------
CROPS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]
PRODUCTS = ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON",
            "EGG", "MILK", "WOOL", "FERTILIZER"]
ANIMALS = ["GOOSE", "COW", "SHEEP"]

SEED_COST = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
CROP_FIRST_YIELD = {"WHEAT": 2, "CARROT": 2, "TOMATO": 8, "STRAWBERRY": 10, "MELON": 10}
ANIMAL_COST = {"GOOSE": 300, "COW": 400, "SHEEP": 500}
ANIMAL_STRUCTURE = {"GOOSE": "COOP", "COW": "PASTURE", "SHEEP": "PASTURE"}
ANIMAL_PRODUCT = {"GOOSE": "EGG", "COW": "MILK", "SHEEP": "WOOL"}

LAND_PRICES = [1000, 2000, 4000]          # NE, SW, SE unlock order
BOARD = 10
SHED_ADJACENT = {(4, 4), (5, 4), (4, 5), (5, 5)}   # (x, y) for boardSize=10
SHED_CAPACITY = 100
MAX_MARKET_ORDERS = 10                    # env: maxMarketOrdersPerTurn
MAX_HANDS = 4                             # policy cap (env itself has no cap)
WHEAT_PICKUP_N = 4                        # units taken by one PICKUP_WHEAT
FARM_HAND_COST_MULT = 1                   # env: farmHandCostMult default

SELLABLE = [p for p in PRODUCTS if p != "FERTILIZER"]   # auto-sell set

# ---- market pricing (exact copy of the env's model, for exact BUY quotes) --
MARKET_I0 = 10000
PRICE_FLOOR = 1
HINGE_GAIN = 8.0
MARKET_PARAMS = {
    "WHEAT":      {"base":  25, "I0": MARKET_I0, "T": 400, "below_func": "sqrt",   "below_target": 0.80, "above_func": "log",    "above_target": 0.20},
    "CARROT":     {"base":  35, "I0": MARKET_I0, "T": 450, "below_func": "hinge",  "below_target": 1.00, "above_func": "sqrt",   "above_target": 0.70},
    "TOMATO":     {"base":  60, "I0": MARKET_I0, "T": 200, "below_func": "hinge",  "below_target": 0.40, "above_func": "sqrt",   "above_target": 0.60},
    "STRAWBERRY": {"base": 120, "I0": MARKET_I0, "T": 100, "below_func": "sqrt",   "below_target": 0.70, "above_func": "linear", "above_target": 1.60},
    "MELON":      {"base": 250, "I0": MARKET_I0, "T": 300, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.60},
    "EGG":        {"base":  50, "I0": MARKET_I0, "T": 332, "below_func": "hinge",  "below_target": 0.40, "above_func": "log",    "above_target": 0.20},
    "MILK":       {"base": 160, "I0": MARKET_I0, "T": 122, "below_func": "sqrt",   "below_target": 0.60, "above_func": "linear", "above_target": 1.60},
    "WOOL":       {"base": 200, "I0": MARKET_I0, "T": 105, "below_func": "log",    "below_target": 0.20, "above_func": "sq",     "above_target": 3.20},
    "FERTILIZER": {"base": 100, "I0": MARKET_I0, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40},
}


def _shape(func, x, T=None):
    x = max(0.0, x)
    if func == "linear": return x
    if func == "sq":     return x * x
    if func == "sqrt":   return math.sqrt(x)
    if func == "log":    return math.log(1.0 + x)
    if func == "log10":  return math.log10(1.0 + x)
    if func == "hinge":
        if not T or T <= 0:
            return x
        u = x / T
        return u + HINGE_GAIN * max(0.0, u - 1.0) ** 2
    return x


def market_price(item, inventory, params=None):
    """Exact replica of the env's pricing (floored at PRICE_FLOOR)."""
    p = (params or MARKET_PARAMS)[item]
    base, I0, T = p["base"], p["I0"], p["T"]
    if inventory < I0:
        amp = p["below_target"] * base / _shape(p["below_func"], T, T)
        price = base + amp * _shape(p["below_func"], I0 - inventory, T)
    else:
        amp = p["above_target"] * base / _shape(p["above_func"], T, T)
        price = base - amp * _shape(p["above_func"], inventory - I0, T)
    return max(PRICE_FLOOR, int(round(price)))


def _fib(n):
    """_fib(0)=1, _fib(1)=1, _fib(2)=2, _fib(3)=3, _fib(4)=5 ..."""
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a + b
    return a


def hire_cost(hires_today, mult=FARM_HAND_COST_MULT):
    return mult * _fib(hires_today)


# --------------------------------------------------------------------------
# Action tables — v2 prefixes FROZEN
# --------------------------------------------------------------------------
FARMER_ACTIONS = (
    ["NORTH", "SOUTH", "EAST", "WEST", "PASS", "WATER", "HARVEST", "DIG"]  # 0-7  (v2)
    + [f"PLANT_{c}" for c in CROPS]                                        # 8-12 (v2)
    + ["FERTILIZE", "PICKUP_FERTILIZER"]                                   # 13-14 (v2)
    # ---- v2.1 additions ---------------------------------------------------
    + ["BUILD_COOP", "BUILD_PASTURE"]                                      # 15-16
    + ["PLACE_GOOSE", "PLACE_COW", "PLACE_SHEEP"]                          # 17-19
    + ["FEED", "CARE", "COLLECT_FERTILIZER"]                               # 20-22
    + ["PICKUP_WHEAT", "PICKUP_GOOSE", "PICKUP_COW", "PICKUP_SHEEP"]       # 23-26
)
MARKET_ACTIONS = (
    ["NOOP"]                                                               # 0    (v2)
    + [f"BUY_SEED_{c}" for c in CROPS]                                     # 1-5  (v2)
    + ["BUY_FERTILIZER", "BUY_LAND"]                                       # 6-7  (v2)
    # ---- v2.1 additions ---------------------------------------------------
    + ["BUY_WHEAT"]                                                        # 8
    + ["BUY_ANIMAL_GOOSE", "BUY_ANIMAL_COW", "BUY_ANIMAL_SHEEP"]           # 9-11
    + ["HIRE"]                                                             # 12
    + [f"SELL_{p}" for p in PRODUCTS]                                      # 13-21 (SELL_FERTILIZER included)
)
N_FARMER = len(FARMER_ACTIONS)     # 27
N_MARKET = len(MARKET_ACTIONS)     # 22

# Built-in agents vary: some take (obs, config), others just (obs). Detect once.
def call_agent(fn, obs, config):
    try:
        n = len(inspect.signature(fn).parameters)
    except (TypeError, ValueError):
        n = 2
    return fn(obs, config) if n >= 2 else fn(obs)

## 2. Unit-perspective encoder — 24 spatial channels, 56 globals

Same board encoding as v2 plus animal channels (one-hot, fed, yield, unfed, fertilizer, cared) and a **unit marker**: `encode_obs2(obs, unit=k)` renders the obs from unit *k*'s perspective (0 = main farmer, 1..4 = hired hands) — its position, its inventory. One shared network serves every unit; the unit index is a global feature.

In [ ]:
# --------------------------------------------------------------------------
# Encoder — 24 spatial channels, 56 globals, unit-perspective
# --------------------------------------------------------------------------
N_SPATIAL = 24
N_GLOBAL = 56
#
# Spatial [24, 10, 10]  (channels 0-14 identical to v2):
#   0  empty unlocked tile            12 plant fertilized_today
#   1  LOCKED tile                    13 COOP structure (occupied or not)
#   2  WEED                           14 PASTURE structure (occupied or not)
#   3-7 crop one-hot (WHEAT..MELON)   15-17 animal one-hot (GOOSE/COW/SHEEP)
#   8  plant age /16 (cap 1)          18 animal fed_today
#   9  plant watered_today            19 animal yield_units /6
#   10 plant consecutive_unwatered/2  20 animal consecutive_unfed /2
#   11 plant yield_units /6           21 animal fertilizer_available
#                                     22 animal cared_today
#                                     23 UNIT MARKER: 1 at this unit's tile
#
# Globals [56]  (indices 0-47 identical to v2, except 3,4,22,23,24,46 are now
# UNIT-relative — identical values when unit=0):
#   0  money /5000                    29 price MELON /250
#   1  day /30                        30 price EGG /250
#   2  hour /24                       31 price MILK /250
#   3  unit x /9                      32 price WOOL /250
#   4  unit y /9                      33 price FERTILIZER /250
#   5-9   seeds /10 (CROPS order)     34-42 (mkt inventory - 10000) /2000
#   10-18 shed products /20 (PRODUCTS order: WHEAT..FERTILIZER)
#   19-21 shed animals /4 (GOOSE/COW/SHEEP)
#   22 UNIT inventory total /10 (cap 2)
#   23 UNIT inv FERTILIZER /5         43 unlocked quadrants /4
#   24 UNIT inv WHEAT /10             44 opponent money /5000
#   25-28 price WHEAT/CARROT/TOMATO/STRAWBERRY /250
#                                     45 town shops /8
#                                     46 unit shed-adjacent flag
#                                     47 hires_today /5
#   ---- v2.1 additions ----
#   48 unit index /4 (0 = main farmer)
#   49 n hands /4
#   50 next hire cost /13
#   51 shed fill fraction (sum of all stacks / 100)
#   52-54 UNIT inv GOOSE/COW/SHEEP /2
#   55 animals placed on farm /8
# NOTE on v2 diff: v2 had shed products at 10-18 in PRODUCTS order too, and
# shed animals at 19-21 — identical. v2's 22-24 were the MAIN FARMER's
# inventory; they are now the CURRENT UNIT's inventory (unit 0 => same value).

def _unit_pos(farm, unit):
    if unit == 0:
        return farm["farmer"]
    if 1 <= unit <= len(farm["hands"]):
        return farm["hands"][unit - 1]
    return None


def encode_obs2(obs, unit=0):
    """Real kaggriculture obs -> (spatial [24,10,10] f32, globals [56] f32)
    from the perspective of `unit` (0 = main farmer, 1..K = hired hands)."""
    player = obs["player"]
    farm = obs["farms"][player]
    opp = obs["farms"][1 - player]
    day = obs["day"]

    s = np.zeros((N_SPATIAL, BOARD, BOARD), dtype=np.float32)
    n_animals = 0
    for y in range(BOARD):
        for x in range(BOARD):
            t = farm["tiles"][y][x]
            if t is None:
                s[0, y, x] = 1.0
                continue
            if t == "LOCKED":
                s[1, y, x] = 1.0
                continue
            if not isinstance(t, dict):
                continue
            kind = t.get("kind")
            if kind == "WEED":
                s[2, y, x] = 1.0
            elif kind == "PLANT":
                s[3 + CROPS.index(t["crop"]), y, x] = 1.0
                s[8, y, x] = min((day - t["planted_day"]) / 16.0, 1.0)
                s[9, y, x] = 1.0 if t["watered_today"] else 0.0
                s[10, y, x] = min(t["consecutive_unwatered"] / 2.0, 1.0)
                s[11, y, x] = min(t["yield_units"] / 6.0, 1.0)
                s[12, y, x] = 1.0 if t.get("fertilized_until_day", -1) >= day else 0.0
            elif kind == "COOP":
                s[13, y, x] = 1.0
            elif kind == "PASTURE":
                s[14, y, x] = 1.0
            if "animal" in t:                       # occupied structure
                n_animals += 1
                s[15 + ANIMALS.index(t["animal"]), y, x] = 1.0
                s[18, y, x] = 1.0 if t["fed_today"] else 0.0
                s[19, y, x] = min(t["yield_units"] / 6.0, 1.0)
                s[20, y, x] = min(t["consecutive_unfed"] / 2.0, 1.0)
                s[21, y, x] = 1.0 if t["fertilizer_available"] else 0.0
                s[22, y, x] = 1.0 if t["cared_today"] else 0.0

    pos = _unit_pos(farm, unit) or farm["farmer"]
    ux, uy = pos[0], pos[1]
    s[23, uy, ux] = 1.0                             # unit marker

    priv = obs["private"]
    shed = priv["shed"]
    invs = priv["inventories"]
    uinv = invs[unit] if unit < len(invs) else {}

    g = np.zeros(N_GLOBAL, dtype=np.float32)
    g[0] = farm["money"] / 5000.0
    g[1] = day / 30.0
    g[2] = obs["hour"] / 24.0
    g[3] = ux / (BOARD - 1)
    g[4] = uy / (BOARD - 1)
    for i, c in enumerate(CROPS):
        g[5 + i] = min(priv["seeds"].get(c, 0) / 10.0, 2.0)
    for i, p in enumerate(PRODUCTS):
        g[10 + i] = min(shed.get(p, 0) / 20.0, 2.0)
    for i, a in enumerate(ANIMALS):
        g[19 + i] = shed.get(a, 0) / 4.0
    g[22] = min(sum(uinv.values()) / 10.0, 2.0)
    g[23] = uinv.get("FERTILIZER", 0) / 5.0
    g[24] = uinv.get("WHEAT", 0) / 10.0
    for i, p in enumerate(PRODUCTS):
        g[25 + i] = obs["market"]["prices"][p] / 250.0
    for i, p in enumerate(PRODUCTS):
        g[34 + i] = (obs["market"]["inventory"][p] - 10000.0) / 2000.0
    g[43] = len(farm["unlocked_quadrants"]) / 4.0
    g[44] = opp["money"] / 5000.0
    g[45] = len(obs["town"]["unlocked_shops"]) / 8.0
    g[46] = 1.0 if (ux, uy) in SHED_ADJACENT else 0.0
    g[47] = min(farm.get("hires_today", 0) / 5.0, 1.0)
    # ---- v2.1 globals ----
    g[48] = unit / 4.0
    g[49] = len(farm["hands"]) / 4.0
    g[50] = min(hire_cost(farm.get("hires_today", 0)) / 13.0, 1.0)
    g[51] = min(sum(shed.values()) / float(SHED_CAPACITY), 1.0)
    for i, a in enumerate(ANIMALS):
        g[52 + i] = min(uinv.get(a, 0) / 2.0, 1.0)
    g[55] = n_animals / 8.0
    return s, g

## 3. Masks & decoders

`action_masks2(obs, unit, auto_sell, wheat_reserve)` — exact legality per the env source: FEED needs the animal + not-fed + wheat **in that unit's inventory**; PLACE needs the matching empty structure + animal in the unit's inventory; HIRE needs `money ≥ fib(hires_today)` and < 4 hands; buys are masked against exact post-buy price quotes; with `auto_sell=True` all SELL_* are masked off (scripted selling), with `False` they're live (learned selling); the last `wheat_reserve` wheat are never sold (feed stock).

`build_action_dict2(f_idx, m_idx, hand_f_idxs, obs, ...)` — emits the real `{"farmer", "hands", "market"}` dict: the main farmer's op, one op per hired hand, the policy's market order first, then (phase 1) scripted auto-sells up to the 10-order cap.

In [ ]:
# --------------------------------------------------------------------------
# Masks — unit-aware, exact legality per env source
# --------------------------------------------------------------------------
def action_masks2(obs, unit=0, auto_sell=True, wheat_reserve=0):
    """-> (farmer_mask [27] bool, market_mask [22] bool) for `unit`.

    auto_sell=True  -> SELL_* masked OFF (curriculum phase 1: scripted selling,
                       build_action_dict2 appends the sells itself).
    auto_sell=False -> SELL_* live (phase 2: learned selling).
    wheat_reserve   -> SELL_WHEAT (and auto-sell) never touch the last
                       `wheat_reserve` WHEAT in the shed (animal feed).
    """
    player = obs["player"]
    farm = obs["farms"][player]
    money = farm["money"]
    seeds = obs["private"]["seeds"]
    shed = obs["private"]["shed"]
    invs = obs["private"]["inventories"]
    uinv = invs[unit] if unit < len(invs) else {}
    mkt = obs["market"]
    params = mkt.get("params") if isinstance(mkt, dict) else None
    shed_full = sum(shed.values()) >= SHED_CAPACITY
    day = obs["day"]

    pos = _unit_pos(farm, unit)
    fm = np.zeros(N_FARMER, dtype=bool)
    if pos is None:                     # no such unit (e.g. hand not hired)
        fm[FARMER_ACTIONS.index("PASS")] = True
    else:
        fx, fy = pos
        tile = farm["tiles"][fy][fx]
        fm[0] = fy > 0
        fm[1] = fy < BOARD - 1
        fm[2] = fx < BOARD - 1
        fm[3] = fx > 0
        fm[4] = True                                                  # PASS

        is_plant = isinstance(tile, dict) and tile.get("kind") == "PLANT"
        is_weed = isinstance(tile, dict) and tile.get("kind") == "WEED"
        is_animal = isinstance(tile, dict) and "animal" in tile
        is_structure = (isinstance(tile, dict)
                        and tile.get("kind") in ("COOP", "PASTURE"))
        shed_adj = (fx, fy) in SHED_ADJACENT

        fm[5] = is_plant and not tile["watered_today"]                # WATER
        fm[6] = ((is_plant and tile["yield_units"] > 0
                  and day - tile["planted_day"] >= CROP_FIRST_YIELD[tile["crop"]])
                 or (is_animal and tile["yield_units"] > 0))          # HARVEST
        fm[7] = (is_plant or is_weed
                 or (is_structure and not is_animal))                 # DIG
        for i, c in enumerate(CROPS):                                 # PLANT_<crop>
            fm[8 + i] = (tile is None) and seeds.get(c, 0) > 0
        fm[13] = (is_plant and uinv.get("FERTILIZER", 0) > 0          # FERTILIZE
                  and tile.get("fertilized_until_day", -1) < day)
        fm[14] = shed_adj and shed.get("FERTILIZER", 0) > 0           # PICKUP_FERTILIZER
        # ---- v2.1 ----
        fm[15] = tile is None                                         # BUILD_COOP (free)
        fm[16] = tile is None                                         # BUILD_PASTURE (free)
        for i, a in enumerate(ANIMALS):                               # PLACE_<animal>
            fm[17 + i] = (is_structure and not is_animal
                          and tile.get("kind") == ANIMAL_STRUCTURE[a]
                          and uinv.get(a, 0) >= 1)
        fm[20] = (is_animal and not tile["fed_today"]                 # FEED
                  and uinv.get("WHEAT", 0) >= 1)
        fm[21] = is_animal and not tile["cared_today"]                # CARE
        fm[22] = is_animal and tile["fertilizer_available"]           # COLLECT_FERTILIZER
        fm[23] = shed_adj and shed.get("WHEAT", 0) > 0                # PICKUP_WHEAT
        for i, a in enumerate(ANIMALS):                               # PICKUP_<animal>
            fm[24 + i] = shed_adj and shed.get(a, 0) > 0

    mm = np.zeros(N_MARKET, dtype=bool)
    mm[0] = True                                                      # NOOP
    for i, c in enumerate(CROPS):                                     # BUY_SEED_<crop>
        mm[1 + i] = money >= SEED_COST[c] and seeds.get(c, 0) < 30
    mm[6] = (money >= market_price("FERTILIZER", mkt["inventory"]["FERTILIZER"] - 1, params)
             and not shed_full)                                       # BUY_FERTILIZER
    n_quads = len(farm["unlocked_quadrants"])
    mm[7] = n_quads < 4 and money >= LAND_PRICES[n_quads - 1]         # BUY_LAND
    # ---- v2.1 ----
    mm[8] = (money >= market_price("WHEAT", mkt["inventory"]["WHEAT"] - 1, params)
             and not shed_full)                                       # BUY_WHEAT
    for i, a in enumerate(ANIMALS):                                   # BUY_ANIMAL_*
        mm[9 + i] = money >= ANIMAL_COST[a] and not shed_full
    mm[12] = (len(farm["hands"]) < MAX_HANDS
              and money >= hire_cost(farm.get("hires_today", 0)))     # HIRE
    if not auto_sell:
        for i, p in enumerate(PRODUCTS):                              # SELL_<product>
            n = shed.get(p, 0) - (wheat_reserve if p == "WHEAT" else 0)
            mm[13 + i] = n > 0
    return fm, mm


# --------------------------------------------------------------------------
# Decoders: action-table index -> real env action dict
# --------------------------------------------------------------------------
def decode_farmer_action(f_idx):
    """Farmer-table index (also used per hand) -> env unit action list."""
    name = FARMER_ACTIONS[f_idx]
    if name.startswith("PLANT_"):
        return ["PLANT", name.split("_", 1)[1]]
    if name.startswith("PLACE_"):
        return ["PLACE", name.split("_", 1)[1]]
    if name == "PICKUP_FERTILIZER":
        return ["PICKUP", "FERTILIZER", 1]
    if name == "PICKUP_WHEAT":
        return ["PICKUP", "WHEAT", WHEAT_PICKUP_N]
    if name.startswith("PICKUP_"):               # PICKUP_GOOSE/COW/SHEEP
        return ["PICKUP", name.split("_", 1)[1], 1]
    return [name]


def decode_market_action(m_idx, obs, wheat_reserve=0):
    """Market-table index -> env market order list (or None for NOOP/no-op)."""
    name = MARKET_ACTIONS[m_idx]
    shed = obs["private"]["shed"]
    if name == "NOOP":
        return None
    if name.startswith("BUY_SEED_"):
        return ["BUY_SEED", name.split("_", 2)[2], 1]
    if name == "BUY_FERTILIZER":
        return ["BUY_PRODUCT", "FERTILIZER", 1]
    if name == "BUY_WHEAT":
        return ["BUY_PRODUCT", "WHEAT", 1]
    if name == "BUY_LAND":
        return ["BUY_LAND"]
    if name.startswith("BUY_ANIMAL_"):
        return ["BUY_ANIMAL", name.split("_", 2)[2], 1]
    if name == "HIRE":
        return ["HIRE"]
    if name.startswith("SELL_"):
        p = name.split("_", 1)[1]
        n = int(shed.get(p, 0)) - (wheat_reserve if p == "WHEAT" else 0)
        return ["SELL", p, n] if n > 0 else None
    raise ValueError(f"unknown market action {m_idx} ({name})")


def build_action_dict2(f_idx, m_idx, hand_f_idxs, obs, auto_sell=True,
                       wheat_reserve=0):
    """(farmer idx, market idx, per-hand farmer idxs, obs) -> env action dict.

    Market: the policy's order goes first; then, if auto_sell, one SELL order
    per shed product (all PRODUCTS except FERTILIZER — now including
    EGG/MILK/WOOL), capped at MAX_MARKET_ORDERS total orders. The last
    `wheat_reserve` WHEAT are never auto-sold (animal feed).
    """
    shed = obs["private"]["shed"]
    action = {
        "farmer": decode_farmer_action(f_idx),
        "hands": [decode_farmer_action(i) for i in (hand_f_idxs or [])],
        "market": [],
    }
    order = decode_market_action(m_idx, obs, wheat_reserve)
    if order is not None:
        action["market"].append(order)
    if auto_sell:
        for p in SELLABLE:
            if len(action["market"]) >= MAX_MARKET_ORDERS:
                break
            n = int(shed.get(p, 0)) - (wheat_reserve if p == "WHEAT" else 0)
            if n > 0:
                action["market"].append(["SELL", p, n])
    return action

### ✅ Contract legality test

Drives a **real env** through the whole new chain: buy goose → pickup → build coop → place → buy wheat → pickup → feed → care → hire ($1) → hand acts → manual SELL. Any malformed action would flip the agent to INVALID — this is the test v1 never had.

In [ ]:
_env = make("kaggriculture", configuration={"episodeSteps": 200, "seed": 3})
_env.reset(num_agents=2)
PASS = {"farmer": ["PASS"], "hands": [], "market": []}

def _obs():
    return _env.state[0].observation

def _play(f_idx, m_idx, hand_idxs=None, auto_sell=False, wheat_reserve=0):
    o = _obs()
    n_hands = len(o["farms"][0]["hands"])
    hands = (hand_idxs if hand_idxs is not None else [FA["PASS"]] * n_hands)[:n_hands]
    a = build_action_dict2(f_idx, m_idx, hands, o, auto_sell=auto_sell, wheat_reserve=wheat_reserve)
    _env.step([a, PASS])
    assert _env.state[0].status == "ACTIVE", f"INVALID action: {a}"
    return a

FA = {n: i for i, n in enumerate(FARMER_ACTIONS)}
MA = {n: i for i, n in enumerate(MARKET_ACTIONS)}

# NOTE: only the NW quadrant (x<5, y<5) is unlocked; (4,4) is the one unlocked
# shed-adjacent tile. Build on (4,3) — (4,5) is LOCKED (SW) and would no-op.
_play(FA["PASS"], MA["BUY_ANIMAL_GOOSE"])            # goose -> shed
assert _obs()["private"]["shed"]["GOOSE"] == 1
_play(FA["PASS"], MA["BUY_WHEAT"])                   # feed wheat -> shed
_play(FA["PICKUP_GOOSE"], MA["NOOP"])                # farmer starts at (4,4), shed-adjacent
_play(FA["PICKUP_WHEAT"], MA["NOOP"])                # wheat -> farmer inventory
_play(FA["NORTH"], MA["NOOP"])                       # (4,3), unlocked NW
_play(FA["BUILD_COOP"], MA["NOOP"])                  # free
_play(FA["PLACE_GOOSE"], MA["NOOP"])                 # goose on the coop
assert _obs()["farms"][0]["tiles"][3][4].get("animal") == "GOOSE"
_play(FA["FEED"], MA["NOOP"])                        # consumes the farmer's wheat
assert _obs()["farms"][0]["tiles"][3][4]["fed_today"]
_play(FA["CARE"], MA["NOOP"])
_play(FA["PASS"], MA["BUY_WHEAT"])                   # one more wheat to test manual selling
_play(FA["PASS"], MA["SELL_WHEAT"], auto_sell=False) # learned-sell path (phase 2)
_play(FA["PASS"], MA["HIRE"])                        # first hand of the day: $1
assert len(_obs()["farms"][0]["hands"]) == 1, "hand not hired"
_play(FA["PASS"], MA["NOOP"], hand_idxs=[FA["PASS"]])  # explicit per-hand action
print("money:", _obs()["farms"][0]["money"], "| hands:", _obs()["farms"][0]["hands"])
print("shed:", {k: v for k, v in _obs()["private"]["shed"].items() if v})
print("ALL CONTRACT CHAIN ACTIONS LEGAL ✓  (goose placed+fed+cared, wheat sold, hand hired)")

## 4. Model & multi-unit action sampling

Same architecture as v2 (conv trunk + globals MLP + shared torso), resized for the v2.1 contract. The new piece is `act_multi`: every env has **U = 1 + max_hands unit slots** (farmer + up to 4 hands). All units are encoded, batched through the network in one forward pass, and each gets its own masked farmer-head action; the market head and value are read from unit 0 only. Slots for hands that don't exist get a PASS-only mask, so they contribute exactly 0 to log-probs and entropy — no gradient noise.

The policy is factorized per env: `log π = Σ_units log π_farmer + log π_market`. Rewards (Δbank) are shared by the whole farm, so all units of an env share the same advantage — standard cooperative multi-agent PPO.

In [ ]:
class FarmPolicy(nn.Module):
    def __init__(self, n_spatial=N_SPATIAL, n_global=N_GLOBAL,
                 n_farmer=N_FARMER, n_market=N_MARKET):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(n_spatial, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.mlp = nn.Sequential(nn.Linear(n_global, 64), nn.ReLU())
        torso_in = 64 * BOARD * BOARD + 64
        self.torso = nn.Sequential(nn.Linear(torso_in, 512), nn.ReLU())
        self.farmer_head = nn.Linear(512, n_farmer)
        self.market_head = nn.Linear(512, n_market)
        self.value_head  = nn.Linear(512, 1)

    def forward(self, spatial, glob):
        z = torch.cat([self.conv(spatial), self.mlp(glob)], dim=1)
        z = self.torso(z)
        return self.farmer_head(z), self.market_head(z), self.value_head(z).squeeze(-1)

def masked_categorical(logits, mask):
    return torch.distributions.Categorical(logits=logits.masked_fill(~mask, -1e9))

@torch.no_grad()
def act_multi(model, S, G, FM, MM, greedy=False):
    """S (N,U,C,H,W), G (N,U,G), FM (N,U,NF), MM (N,NM) ->
    fa (N,U), ma (N,), logp (N,), value (N,), entropy (N,)."""
    N, U = S.shape[0], S.shape[1]
    f_logits, m_logits, v = model(S.reshape(N * U, *S.shape[2:]).to(device),
                                  G.reshape(N * U, G.shape[2]).to(device))
    f_logits = f_logits.view(N, U, -1)
    m_logits0 = m_logits.view(N, U, -1)[:, 0]     # market head: unit 0 only
    v0 = v.view(N, U)[:, 0]                        # value: unit 0 only
    fd = masked_categorical(f_logits, FM.to(device))
    md = masked_categorical(m_logits0, MM.to(device))
    if greedy:
        fa = fd.logits.argmax(-1)
        ma = md.logits.argmax(-1)
    else:
        fa = fd.sample()
        ma = md.sample()
    logp = fd.log_prob(fa).sum(-1) + md.log_prob(ma)
    ent = fd.entropy().sum(-1) + md.entropy()
    return fa.cpu(), ma.cpu(), logp.cpu(), v0.cpu(), ent.cpu()

_m = FarmPolicy().to(device)
print(f"parameters: {sum(p.numel() for p in _m.parameters()):,}  (U = 1 + {CONFIG['max_hands']} unit slots/env)")
del _m

## 5. The self-play league

* **`SnapshotOpponent`** — a frozen copy of the policy, playable as a kaggle agent in *either* seat (it reads `obs["player"]`). It runs the same multi-unit inference as the learner, so snapshots know how to use hands and animals. Picklable-safe: ships only the CPU `state_dict`; the module is rebuilt lazily per process (important for the multiprocessing workers).
* **`SnapshotLeague`** — a weighted pool of `"starter"` / `"random"` / snapshots. Fresh opponent per episode; per-opponent results tracked for split logging ("final$ vs starter / vs snapshot_ep60 / ...").
* **`Curriculum`** — per-epoch settings. Phase 1: scripted auto-sell, starter/random only. Phase 2 (from `phase2_epoch`): learned selling (`auto_sell=False`), snapshots frozen into the pool every `snapshot_every` epochs.

In [ ]:
class SnapshotOpponent:
    """Frozen FarmPolicy as a kaggle-environments agent (multi-unit, v2.1 contract)."""
    def __init__(self, state_dict, auto_sell=True, wheat_reserve=0, greedy=True):
        self.state_dict = {k: v.detach().cpu().clone() for k, v in state_dict.items()}
        self.auto_sell = auto_sell
        self.wheat_reserve = wheat_reserve
        self.greedy = greedy
        self._model = None

    def __getstate__(self):                    # pickle only the state_dict
        d = self.__dict__.copy(); d["_model"] = None
        return d

    def _net(self):
        if self._model is None:
            m = FarmPolicy()
            m.load_state_dict(self.state_dict)
            m.eval()
            self._model = m
        return self._model

    @torch.no_grad()
    def __call__(self, obs, config=None):
        farm = obs["farms"][obs["player"]]
        n_hands = len(farm["hands"])
        S, G, FM = [], [], []
        for u in range(1 + n_hands):
            s, g = encode_obs2(obs, unit=u)
            fm, mm = action_masks2(obs, unit=u, auto_sell=self.auto_sell,
                                   wheat_reserve=self.wheat_reserve)
            S.append(s); G.append(g); FM.append(fm)
        model = self._net()
        f_logits, m_logits, _ = model(torch.tensor(np.stack(S)), torch.tensor(np.stack(G)))
        f_logits = f_logits.masked_fill(~torch.tensor(np.stack(FM)), -1e9)
        m_logits = m_logits[0].masked_fill(~torch.tensor(mm), -1e9)
        if self.greedy:
            fa = f_logits.argmax(-1).tolist()
            ma = int(m_logits.argmax(-1))
        else:
            fa = torch.distributions.Categorical(logits=f_logits).sample().tolist()
            ma = int(torch.distributions.Categorical(logits=m_logits).sample())
        return build_action_dict2(fa[0], ma, fa[1:], obs,
                                  auto_sell=self.auto_sell, wheat_reserve=self.wheat_reserve)

class SnapshotLeague:
    def __init__(self, mix, max_snapshots=8, auto_sell=True, wheat_reserve=0):
        self.mix = dict(mix)
        self.max_snapshots = max_snapshots
        self.auto_sell = auto_sell
        self.wheat_reserve = wheat_reserve
        self.snapshot_every = 0
        self.snapshots = []          # (tag, state_dict), oldest first
        self.results = {}            # tag -> [learner final$]

    def add(self, state_dict, tag):
        self.snapshots.append((tag, {k: v.detach().cpu().clone() for k, v in state_dict.items()}))
        while len(self.snapshots) > self.max_snapshots:
            self.snapshots.pop(0)

    def should_snapshot(self, done_epochs):
        return self.snapshot_every > 0 and done_epochs % self.snapshot_every == 0

    def sample(self, rng):
        tags = list(self.mix.keys())
        w = np.array([self.mix[t] for t in tags], dtype=np.float64)
        if not self.snapshots and "snapshot" in self.mix:      # fold into starter until pool exists
            i = tags.index("snapshot")
            w[tags.index("starter")] += w[i]; w[i] = 0.0
        w = w / w.sum()
        kind = tags[rng.choice(len(tags), p=w)]
        if kind == "snapshot":
            tag, sd = self.snapshots[rng.integers(len(self.snapshots))]
            return ("snapshot", tag, SnapshotOpponent(sd, self.auto_sell, self.wheat_reserve))
        return (kind, kind, kind)

    def record(self, tag, final_money):
        self.results.setdefault(tag, []).append(float(final_money))

    def summary(self):
        return {t: (len(v), float(np.mean(v))) for t, v in sorted(self.results.items())}

class Phase:
    def __init__(self, until_epoch, auto_sell, wheat_reserve, mix, snapshot_every):
        self.until_epoch, self.auto_sell, self.wheat_reserve = until_epoch, auto_sell, wheat_reserve
        self.mix, self.snapshot_every = mix, snapshot_every

class Curriculum:
    def __init__(self, phases):
        self.phases = sorted(phases, key=lambda p: p.until_epoch)
    def settings(self, epoch):
        for p in self.phases:
            if epoch < p.until_epoch:
                return p
        raise ValueError("need a final phase with until_epoch=inf")

def build_curriculum(cfg):
    return Curriculum([
        Phase(cfg["phase2_epoch"], True, cfg["wheat_reserve"], cfg["league_mix_p1"], 0),
        Phase(math.inf, False, cfg["wheat_reserve"], cfg["league_mix_p2"], cfg["snapshot_every"]),
    ])

## 6. VecKag21 — league-driven, multi-unit vectorized runner

Per step it emits `(S, G, FM, MM)` with shapes `(N, U, 24, 10, 10)`, `(N, U, 56)`, `(N, U, 27)`, `(N, 22)`; `step()` takes `(fa (N,U), ma (N,))`. Reward = Δbank × scale. Opponents are re-sampled from the league at every episode reset (fresh seed per episode), results recorded per opponent tag. `set_sell_mode()` is the curriculum hook.

Note for BC: this in-process runner exposes `.envs` (the multiprocessing runner does not — BC collection always uses this one).

In [ ]:
class VecKag21:
    def __init__(self, n_envs, cfg, league, seed=0):
        self.n = n_envs
        self.cfg = cfg
        self.U = 1 + cfg["max_hands"]
        self.league = league
        self.rng = np.random.default_rng(seed)
        self.auto_sell = True
        self.wheat_reserve = cfg["wheat_reserve"]
        self.episode_count = [0] * n_envs
        self.opponents = [None] * n_envs
        self.opp_tags = ["<none>"] * n_envs
        self.envs = [None] * n_envs
        self.prev_money = np.zeros(n_envs, dtype=np.float64)
        self.final_moneys = []
        self._opp_cache = {}

    def set_sell_mode(self, auto_sell, wheat_reserve=None):
        self.auto_sell = auto_sell
        if wheat_reserve is not None:
            self.wheat_reserve = wheat_reserve

    def _opp_fn(self, env, spec):
        if not isinstance(spec, str):          # SnapshotOpponent: fresh object, don't cache
            return spec
        if spec not in self._opp_cache:
            self._opp_cache[spec] = env.agents[spec]
        return self._opp_cache[spec]

    def _make_env(self, i):
        kind, tag, opp = self.league.sample(self.rng)
        self.opponents[i] = opp
        self.opp_tags[i] = tag
        conf = {"episodeSteps": self.cfg["episode_steps"],
                "seed": 1000 * self.episode_count[i] + 7 * i + 1}
        env = make("kaggriculture", configuration=conf)
        env.reset(num_agents=2)
        return env

    def reset(self):
        for i in range(self.n):
            self.envs[i] = self._make_env(i)
            self.episode_count[i] += 1
        self.prev_money[:] = self.cfg["starting_money"]
        return self._observe_all()

    def _observe_one(self, i):
        obs = self.envs[i].state[0].observation
        n_hands = len(obs["farms"][obs["player"]]["hands"])
        S = np.zeros((self.U, N_SPATIAL, BOARD, BOARD), dtype=np.float32)
        G = np.zeros((self.U, N_GLOBAL), dtype=np.float32)
        FM = np.zeros((self.U, N_FARMER), dtype=bool)
        _, mm0 = action_masks2(obs, unit=0, auto_sell=self.auto_sell,
                               wheat_reserve=self.wheat_reserve)   # market mask is unit-independent
        for u in range(self.U):
            if u <= n_hands:
                s, g = encode_obs2(obs, unit=u)
                fm, _ = action_masks2(obs, unit=u, auto_sell=self.auto_sell,
                                      wheat_reserve=self.wheat_reserve)
                S[u], G[u], FM[u] = s, g, fm
            else:
                FM[u, FARMER_ACTIONS.index("PASS")] = True     # dead slot: PASS-only
        return S, G, FM, mm0

    def _observe_all(self):
        S, G, FM, MM = zip(*[self._observe_one(i) for i in range(self.n)])
        return (torch.tensor(np.stack(S)), torch.tensor(np.stack(G)),
                torch.tensor(np.stack(FM)), torch.tensor(np.stack(MM)))

    def step(self, f_actions, m_actions):
        rewards = np.zeros(self.n, dtype=np.float32)
        dones = np.zeros(self.n, dtype=bool)
        for i, env in enumerate(self.envs):
            obs0 = env.state[0].observation
            n_hands = len(obs0["farms"][obs0["player"]]["hands"])
            hand_idxs = [int(f_actions[i, u]) for u in range(1, 1 + n_hands)]
            a0 = build_action_dict2(int(f_actions[i, 0]), int(m_actions[i]), hand_idxs, obs0,
                                    auto_sell=self.auto_sell, wheat_reserve=self.wheat_reserve)
            try:
                a1 = call_agent(self._opp_fn(env, self.opponents[i]),
                                env.state[1].observation, env.configuration)
                if not isinstance(a1, dict):
                    a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            except Exception:
                a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            env.step([a0, a1])
            money = env.state[0].observation["farms"][0]["money"]
            rewards[i] = (money - self.prev_money[i]) * self.cfg["reward_scale"]
            self.prev_money[i] = money
            if env.state[0].status != "ACTIVE":
                dones[i] = True
                self.final_moneys.append(money)
                self.league.record(self.opp_tags[i], money)
                self.envs[i] = self._make_env(i)
                self.episode_count[i] += 1
                self.prev_money[i] = self.cfg["starting_money"]
        return self._observe_all(), torch.tensor(rewards), torch.tensor(dones, dtype=torch.float32)

### ✅ Runner test

Random *masked* multi-unit actions for a few steps, then a full legality check of a snapshot opponent playing a complete game as player 1.

In [ ]:
_league = SnapshotLeague(CONFIG["league_mix_p1"], CONFIG["max_snapshots"])
_vec = VecKag21(CONFIG["n_envs"], CONFIG, _league)
_S, _G, _FM, _MM = _vec.reset()
print("batch:", _S.shape, _G.shape, _FM.shape, _MM.shape)
for _t in range(4):
    _fa = torch.stack([torch.tensor([np.random.choice(np.where(m.numpy())[0]) for m in u]) for u in _FM])
    _ma = torch.stack([torch.tensor(np.random.choice(np.where(m.numpy())[0])) for m in _MM])
    (_S, _G, _FM, _MM), _r, _d = _vec.step(_fa, _ma)
    print(f"t={_t}  reward={_r.numpy().round(2).tolist()}  done={_d.tolist()}")
del _vec

# snapshot opponent legality: untrained net, full short game as player 1
_snap = SnapshotOpponent(FarmPolicy().state_dict())
_env = make("kaggriculture", configuration={"episodeSteps": 200, "seed": 11})
_env.reset(num_agents=2)
_steps = 0
while _env.state[0].status == "ACTIVE" and _steps < 200:
    _a0 = call_agent(_env.agents["starter"], _env.state[0].observation, _env.configuration)
    _a1 = _snap(_env.state[1].observation, _env.configuration)
    _env.step([_a0, _a1])
    _steps += 1
print(f"snapshot opponent played {_steps} steps, status={_env.state[1].status} (no INVALID) ✓")

## 7. Behavior cloning — the winner recipe

This is the fix that took the greedy BC policy from **$3,000 (does nothing) to $3,834 (beats the $3,586 expert)**, found by systematic ablation. Plain cross-entropy fails because the expert's labels are 93% PASS / 98.4% NOOP: the pivotal `BUY iff seeds==0` rule sits on 1.6% of frames, and CE drives its probability to literal zero. Three ingredients, all needed:

1. **Market-head per-sample weights** — BUY frames ×20, NOOP boundary frames ×50 (farmer head stays plain CE).
2. **One DAgger round** — roll the BC policy greedily, let the expert relabel every state it *actually visits*. This supplies the missing NOOP labels on the out-of-distribution "seed stockpile" states the naive policy wanders into. (A second DAgger round degrades the farmer — stop at one.)
3. **Deep Supervision Logit Adjustment** — class-prior correction folded into the head biases, *present classes only*: `bias_c -= tau·log(count_c/N)` with **τ_farmer=1.5, τ_market=0.3**. τ_m=0.3 (not 1.0!) places the BUY/NOOP threshold between the true-buy and erroneous-buy logit margins; the plateau is wide (0.25–0.5).

The starter expert never hires hands, so hand slots are labeled PASS under PASS-only masks — hands are PPO's job to discover.

In [ ]:
def expert_labels(act_d):
    """starter's action dict -> (farmer_idx, market_idx). v2.1 indices 0-14/0-7 are frozen, so this is unchanged from v2."""
    fop = act_d.get("farmer", ["PASS"])
    fi = 4                                                    # PASS
    if fop[0] in ("NORTH", "SOUTH", "EAST", "WEST", "WATER", "HARVEST", "DIG", "FERTILIZE"):
        fi = FARMER_ACTIONS.index(fop[0])
    elif fop[0] == "PLANT" and len(fop) > 1 and fop[1] in CROPS:
        fi = 8 + CROPS.index(fop[1])
    elif fop[0] == "PICKUP" and len(fop) > 1 and fop[1] == "FERTILIZER":
        fi = FARMER_ACTIONS.index("PICKUP_FERTILIZER")
    mi = 0
    for order in act_d.get("market", []):
        if order[0] == "BUY_SEED" and order[1] in CROPS:
            mi = 1 + CROPS.index(order[1]); break
        if order[0] == "BUY_PRODUCT" and order[1] == "FERTILIZER":
            mi = 6; break
        if order[0] == "BUY_LAND":
            mi = 7; break
    return fi, mi

def collect_bc_data(vec, n_episodes_per_env):
    """Roll envs forward playing the expert; label every step (unit 0 = expert, hands = PASS)."""
    starter = None
    S_b, G_b, FM_b, MM_b, FA_b, MA_b = [], [], [], [], [], []
    S, G, FM, MM = vec.reset()
    PASS_I = FARMER_ACTIONS.index("PASS")
    for _ in range(n_episodes_per_env * vec.cfg["episode_steps"]):
        fa = torch.full((vec.n, vec.U), PASS_I, dtype=torch.long)
        ma = torch.zeros(vec.n, dtype=torch.long)
        for i, env in enumerate(vec.envs):
            if starter is None:
                starter = env.agents["starter"]
            fi, mi = expert_labels(call_agent(starter, env.state[0].observation, env.configuration))
            fa[i, 0], ma[i] = fi, mi
        S_b.append(S); G_b.append(G); FM_b.append(FM); MM_b.append(MM)
        FA_b.append(fa); MA_b.append(ma)
        (S, G, FM, MM), _, _ = vec.step(fa, ma)
    return (torch.stack(S_b), torch.stack(G_b), torch.stack(FM_b),
            torch.stack(MM_b), torch.stack(FA_b), torch.stack(MA_b))

def behavior_clone(model, vec, cfg):
    BUY = MARKET_ACTIONS.index("BUY_SEED_CARROT")
    PASS_I = FARMER_ACTIONS.index("PASS")

    def flatten(data):
        S, G, FM, MM, FA, MA = data
        T, N = S.shape[0], S.shape[1]
        return ((S.reshape(T * N * vec.U, N_SPATIAL, BOARD, BOARD).to(device),
                 G.reshape(T * N * vec.U, N_GLOBAL).to(device),
                 FM.reshape(T * N * vec.U, N_FARMER).to(device),
                 FA.reshape(-1).to(device)),
                (MM.reshape(T * N, N_MARKET).to(device),
                 MA.reshape(-1).to(device)))

    def fit(fdata, mdata, sample_w_mul=None):
        bS, bG, bFM, bFA = fdata
        bMM, bMA = mdata
        torch.manual_seed(0); np.random.seed(0)
        m = FarmPolicy().to(device)                  # fresh net, same seed each fit
        opt = torch.optim.Adam(m.parameters(), lr=cfg["bc_lr"])
        w_m = torch.ones(bMA.shape[0], device=device)
        w_m[bMA == BUY] = cfg["bc_w_buy"]
        w_m[bMA == 0] = cfg["bc_w_noop"]
        if sample_w_mul is not None:
            w_m = w_m * sample_w_mul
        m.train()
        E = bMM.shape[0]                            # env-samples (unit blocks of size vec.U)
        ar = torch.arange(vec.U, device=device)
        for ep in range(cfg["bc_epochs"]):
            perm = torch.randperm(E, device=device)
            tot_l = cnt = 0
            for st in range(0, E, 1024):
                eidx = perm[st:st + 1024]
                uidx = (eidx.unsqueeze(1) * vec.U + ar).view(-1)   # all units of those envs
                f_logits, m_logits, _ = m(bS[uidx], bG[uidx])
                fl = f_logits.masked_fill(~bFM[uidx], -1e9)
                loss_f = F.cross_entropy(fl, bFA[uidx])
                # market head: unit-0 slice only
                m0 = m_logits.view(-1, vec.U, N_MARKET)[:, 0]
                ml = m0.masked_fill(~bMM[eidx], -1e9)
                loss_m = (F.cross_entropy(ml, bMA[eidx], reduction="none") * w_m[eidx]).mean()
                loss = loss_f + loss_m
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(m.parameters(), cfg["max_grad_norm"])
                opt.step()
                tot_l += loss.item(); cnt += 1
            print(f"    fit epoch {ep}: loss {tot_l/cnt:.4f}")
        return m

    @torch.no_grad()
    def adjust(m, bFA, bMA):
        for head, y, C, tau in ((m.farmer_head, bFA, N_FARMER, cfg["bc_tau_farmer"]),
                                (m.market_head, bMA, N_MARKET, cfg["bc_tau_market"])):
            counts = torch.bincount(y, minlength=C).float()
            adj = torch.zeros(C, device=counts.device)
            p = counts > 0                            # PRESENT classes only!
            adj[p] = torch.log(counts[p] / y.shape[0])
            head.bias -= tau * adj.to(head.bias.device)

    @torch.no_grad()
    def dagger_rollout(m, n_steps):
        starter = None
        S_b, G_b, FM_b, MM_b, FA_b, MA_b = [], [], [], [], [], []
        S, G, FM, MM = vec.reset()
        m.eval()
        for _ in range(n_steps):
            fa_lab = torch.full((vec.n, vec.U), PASS_I, dtype=torch.long)
            ma_lab = torch.zeros(vec.n, dtype=torch.long)
            for i, env in enumerate(vec.envs):
                if starter is None:
                    starter = env.agents["starter"]
                fi, mi = expert_labels(call_agent(starter, env.state[0].observation, env.configuration))
                fa_lab[i, 0], ma_lab[i] = fi, mi
            S_b.append(S); G_b.append(G); FM_b.append(FM); MM_b.append(MM)
            FA_b.append(fa_lab); MA_b.append(ma_lab)
            pa, pma, _, _, _ = act_multi(m, S, G, FM, MM, greedy=True)
            (S, G, FM, MM), _, _ = vec.step(pa, pma)
        return (torch.stack(S_b), torch.stack(G_b), torch.stack(FM_b),
                torch.stack(MM_b), torch.stack(FA_b), torch.stack(MA_b))

    print("  BC: collecting expert data ...")
    t0 = time.time()
    fdata0, mdata0 = flatten(collect_bc_data(vec, cfg["bc_episodes"]))
    print(f"  {fdata0[0].shape[0]} unit frames in {time.time()-t0:.0f}s")
    m = fit(fdata0, mdata0)
    adjust(m, fdata0[3], mdata0[1])
    for r in range(cfg["bc_dagger_rounds"]):
        print(f"  BC: DAgger round {r+1} ...")
        fd1, md1 = flatten(dagger_rollout(m, cfg["episode_steps"]))
        fdata = [torch.cat(ts) for ts in zip(fdata0, fd1)]
        mdata = [torch.cat(ts) for ts in zip(mdata0, md1)]
        mul = torch.ones(mdata[1].shape[0], device=device)
        mul[mdata0[1].shape[0]:] = cfg["bc_w_dagger"]
        m = fit(fdata, mdata, sample_w_mul=mul)
        adjust(m, fdata[3], mdata[1])
    model.load_state_dict(m.state_dict())
    path = os.path.join(CKPT_DIR, "bc_init.pt")
    torch.save(model.state_dict(), path)
    print("  saved", path)

### ▶️ Create model + league + runner, run BC

After BC, the greedy policy should score **≈ $3,800** (expert level) on full seasons. In SMOKE mode (4-day seasons, 1 expert episode) expect less — the check is that the greedy policy *buys seeds and farms* (nonzero BUY/PLANT counts), not the dollar figure.

In [ ]:
model = FarmPolicy().to(device)
league = SnapshotLeague(CONFIG["league_mix_p1"], CONFIG["max_snapshots"],
                        auto_sell=True, wheat_reserve=CONFIG["wheat_reserve"])
vec = VecKag21(CONFIG["n_envs"], CONFIG, league)

if CONFIG["use_bc_init"]:
    behavior_clone(model, vec, CONFIG)
    vec.final_moneys.clear()          # remove expert episodes from the PPO metric (v2 logging bug)
    # frozen BC reference for the KL anchor (captured BEFORE temperature scaling)
    anchor_model = FarmPolicy().to(device)
    anchor_model.load_state_dict(model.state_dict())
else:
    anchor_model = None
    print("BC skipped")

### ✅ Greedy eval of the BC policy

Multi-unit greedy inference, full seasons, with action histograms — the histogram is the quick sanity check: you want to see BUY_SEED / PLANT / WATER / HARVEST firing, not 100% PASS+NOOP.

In [ ]:
from collections import Counter

@torch.no_grad()
def evaluate(model, opponent="starter", n_games=5, render_first=False, verbose=True):
    model.eval()
    results, acts_f, acts_m = [], Counter(), Counter()
    for game in range(n_games):
        env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 5000 + game})
        env.reset(num_agents=2)
        opp = env.agents[opponent] if isinstance(opponent, str) else opponent
        while env.state[0].status == "ACTIVE":
            obs0 = env.state[0].observation
            n_hands = len(obs0["farms"][0]["hands"])
            S, G, FM = [], [], []
            for u in range(1 + n_hands):
                s, g = encode_obs2(obs0, unit=u)
                fm, mm = action_masks2(obs0, unit=u, auto_sell=True,
                                       wheat_reserve=CONFIG["wheat_reserve"])
                S.append(s); G.append(g); FM.append(fm)
            f_logits, m_logits, _ = model(torch.tensor(np.stack(S)).to(device),
                                          torch.tensor(np.stack(G)).to(device))
            f_logits = f_logits.masked_fill(~torch.tensor(np.stack(FM)).to(device), -1e9)
            m_logits = m_logits[0].masked_fill(~torch.tensor(mm).to(device), -1e9)
            fa = f_logits.argmax(-1).cpu().tolist()
            ma = int(m_logits.argmax(-1).cpu())
            for u in fa: acts_f[FARMER_ACTIONS[u]] += 1
            acts_m[MARKET_ACTIONS[ma]] += 1
            a0 = build_action_dict2(fa[0], ma, fa[1:], obs0, auto_sell=True,
                                    wheat_reserve=CONFIG["wheat_reserve"])
            try:
                a1 = call_agent(opp, env.state[1].observation, env.configuration)
                if not isinstance(a1, dict):
                    a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            except Exception:
                a1 = {"farmer": ["PASS"], "hands": [], "market": []}
            env.step([a0, a1])
        results.append(env.state[0].observation["farms"][0]["money"])
        if game == 0 and render_first:
            out = os.path.join(WORKDIR, f"replay_vs_{opponent if isinstance(opponent, str) else 'snapshot'}.html")
            with open(out, "w") as f:
                f.write(env.render(mode="html"))
            print("  replay:", out)
    if verbose:
        tag = opponent if isinstance(opponent, str) else "snapshot"
        print(f"vs {tag:8s}: mean ${np.mean(results):.0f} (min ${min(results):.0f}, max ${max(results):.0f})")
        print(f"  farmer: {acts_f.most_common(8)}")
        print(f"  market: {acts_m.most_common(5)}")
    return results

if SMOKE:
    print("(SMOKE: 1 game — checking the histogram, not the score)")
    evaluate(model, "starter", n_games=1)
else:
    evaluate(model, "starter", n_games=5, render_first=True)

## 8. Post-BC temperature calibration

The BC winner is extremely peaked (≈0.25 nats) — fed straight into PPO it collapses (ratios ≈ 1, KL ≈ 0, no exploration; the v2 medium run's failure). Dividing both actor heads' final-layer weights and biases by a temperature T reopens exploration *without changing what the network knows*. T is grid-searched so the mean sampled entropy hits `target_entropy`.

**Target 0.4 nats, not 0.8**: the v2.1 medium run showed that overheating a *good* init (T≈5 for 0.8 nats) crashes sampled performance from $3,886 to ~$1,100 and lets the greedy mode drift to do-nothing. 0.4 nats (T≈3) keeps the warm start mostly intact while still giving PPO room to explore. The BC-KL anchor in the PPO cell is the second layer of protection.

In [ ]:
def set_head_temperature(model, T):
    with torch.no_grad():
        for head in (model.farmer_head, model.market_head):
            head.weight.div_(T)
            head.bias.div_(T)

@torch.no_grad()
def calibrate_temperature(model, vec, target):
    S, G, FM, MM = vec.reset()
    Ss, Gs, FMs, MMs = [], [], [], []
    for _ in range(60):
        Ss.append(S); Gs.append(G); FMs.append(FM); MMs.append(MM)
        fa, ma, *_ = act_multi(model, S, G, FM, MM)
        (S, G, FM, MM), _, _ = vec.step(fa, ma)
    bS = torch.cat(Ss).to(device); bG = torch.cat(Gs).to(device)
    bFM = torch.cat(FMs).to(device); bMM = torch.cat(MMs).to(device)
    N, U = bS.shape[0], bS.shape[1]
    f_logits, m_logits, _ = model(bS.view(N * U, N_SPATIAL, BOARD, BOARD),
                                  bG.view(N * U, N_GLOBAL))
    f0 = f_logits.view(N, U, -1)[:, 0]
    m0 = m_logits.view(N, U, -1)[:, 0]

    def ent_at(T):
        fd = masked_categorical(f0 / T, bFM[:, 0])
        md = masked_categorical(m0 / T, bMM)
        return (fd.entropy() + md.entropy()).mean().item()

    ents = [(T, ent_at(T)) for T in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0]]
    print("  temperature grid:", " ".join(f"T={T}:{e:.2f}" for T, e in ents))
    return min(ents, key=lambda te: abs(te[1] - target))

if CONFIG["use_bc_init"] and CONFIG["target_entropy"] > 0:
    Tb, eb = calibrate_temperature(model, vec, CONFIG["target_entropy"])
    print(f"  applying head temperature T={Tb} (entropy {eb:.2f} -> target {CONFIG['target_entropy']})")
    set_head_temperature(model, Tb)
    vec.final_moneys.clear()          # calibration episodes are not policy episodes either

## 9. PPO v2.1

All the v2 machinery (GAE, stored masks re-applied in the update, grad clipping, Δbank reward) plus the three wave-1 fixes and two v2.1 features:

* **Entropy schedule** `ent_coef_start → ent_coef_end` (linear) — no more post-BC collapse;
* **Value warmup** — first `value_warmup_epochs` epochs train the value head only (actor frozen), so advantages are calibrated before the policy moves;
* **BC-KL anchor** — a KL penalty to the frozen post-BC policy, annealed `anchor_coef_start → 0` over `anchor_epochs`. The medium run showed why this is needed: temperature + entropy bonus alone let the *mode* drift to do-nothing (greedy $3,000) even while the *mixture* earned $3,200;
* **In-training greedy checks** — a 1-game greedy eval every `greedy_check_every` epochs, printed as `GREEDY $…`, so mode collapse shows up in the log immediately;
* **Curriculum** — phase 1 scripted auto-sell vs starter/random; phase 2 learned selling + **league snapshots** frozen every `snapshot_every` epochs;
* **Per-opponent logging** — `vs[starter:…(n=…) | snapshot_ep60:…(n=…)]` each epoch;
* multi-unit update: farmer head flattens all unit slots, market head + value read unit 0, all units of an env share the env's advantage.

In [ ]:
def train_ppo(model, vec, cfg, league, curriculum, anchor_model=None, greedy_fn=None):
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    T, N, U = cfg["episode_steps"], vec.n, vec.U
    hist = {"final_money": [], "entropy": [], "kl": [], "vloss": [], "vs_opponent": [], "greedy": []}
    odt = cfg["obs_dtype"]
    if anchor_model is not None:
        anchor_model.eval()
        for p in anchor_model.parameters():
            p.requires_grad = False

    S, G, FM, MM = vec.reset()
    for epoch in range(cfg["epochs"]):
        t0 = time.time()
        cs = curriculum.settings(epoch)
        vec.set_sell_mode(cs.auto_sell, cs.wheat_reserve)
        league.mix = cs.mix
        league.snapshot_every = cs.snapshot_every
        ent_coef = cfg["ent_coef_start"] + (cfg["ent_coef_end"] - cfg["ent_coef_start"]) * epoch / max(1, cfg["epochs"] - 1)
        anchor_coef = 0.0
        if anchor_model is not None:
            frac = min(1.0, epoch / max(1, cfg["anchor_epochs"]))
            anchor_coef = cfg["anchor_coef_start"] + (cfg["anchor_coef_end"] - cfg["anchor_coef_start"]) * frac
        warming = epoch < cfg["value_warmup_epochs"]
        for nme, p in model.named_parameters():
            p.requires_grad = (nme.startswith("value_head") or not warming)

        sp  = torch.zeros(T, N, U, N_SPATIAL, BOARD, BOARD, dtype=odt)
        gl  = torch.zeros(T, N, U, N_GLOBAL, dtype=odt)
        fmk = torch.zeros(T, N, U, N_FARMER, dtype=torch.bool)
        mmk = torch.zeros(T, N, N_MARKET, dtype=torch.bool)
        fac = torch.zeros(T, N, U, dtype=torch.long)
        mac = torch.zeros(T, N, dtype=torch.long)
        lgp = torch.zeros(T, N)
        val = torch.zeros(T, N)
        rew = torch.zeros(T, N)
        don = torch.zeros(T, N)

        model.eval()
        for t in range(T):
            sp[t], gl[t], fmk[t], mmk[t] = S.to(odt), G.to(odt), FM, MM
            fa, ma, logp, v, _ = act_multi(model, S, G, FM, MM)
            fac[t], mac[t], lgp[t], val[t] = fa, ma, logp, v
            (S, G, FM, MM), r, d = vec.step(fa, ma)
            rew[t], don[t] = r, d

        # ---- GAE with bootstrap -------------------------------------------
        with torch.no_grad():
            _, _, v_last = model(S[:, 0].to(device), G[:, 0].to(device))
        v_last = v_last.cpu()
        adv = torch.zeros(T, N)
        lastgae = torch.zeros(N)
        for t in reversed(range(T)):
            nonterm = 1.0 - don[t]
            nextv = v_last if t == T - 1 else val[t + 1]
            delta = rew[t] + cfg["gamma"] * nextv * nonterm - val[t]
            lastgae = delta + cfg["gamma"] * cfg["lam"] * nonterm * lastgae
            adv[t] = lastgae
        ret = adv + val

        # ---- PPO update ----------------------------------------------------
        B = T * N
        bsp, bgl = sp.view(B, U, N_SPATIAL, BOARD, BOARD), gl.view(B, U, N_GLOBAL)
        bfmk, bmmk = fmk.view(B, U, N_FARMER), mmk.view(B, N_MARKET)
        bfac, bmac, blgp = fac.view(B, U), mac.view(-1), lgp.view(-1)
        badv, bret = adv.view(-1), ret.view(-1)
        badv = (badv - badv.mean()) / (badv.std() + 1e-8)

        model.train()
        ep_ent = ep_kl = ep_vl = nb = 0
        for _ in range(cfg["ppo_epochs"]):
            perm = torch.randperm(B)
            for st in range(0, B, cfg["mini_batch"]):
                idx = perm[st:st + cfg["mini_batch"]]
                mb = idx.shape[0]
                f_logits, m_logits, v = model(bsp[idx].view(mb * U, N_SPATIAL, BOARD, BOARD).to(device).float(),
                                              bgl[idx].view(mb * U, N_GLOBAL).to(device).float())
                f_logits = f_logits.view(mb, U, N_FARMER)
                m_logits = m_logits.view(mb, U, N_MARKET)[:, 0]
                v0 = v.view(mb, U)[:, 0]
                fd = masked_categorical(f_logits, bfmk[idx].to(device))
                md = masked_categorical(m_logits, bmmk[idx].to(device))
                new_logp = fd.log_prob(bfac[idx].to(device)).sum(-1) + md.log_prob(bmac[idx].to(device))
                ratio = torch.exp(new_logp - blgp[idx].to(device))
                a_i = badv[idx].to(device)
                surr = torch.min(ratio * a_i,
                                 torch.clamp(ratio, 1 - cfg["clip_eps"], 1 + cfg["clip_eps"]) * a_i)
                actor_loss = -surr.mean()
                v_loss = F.mse_loss(v0, bret[idx].to(device))
                ent = (fd.entropy().sum(-1) + md.entropy()).mean()
                if warming:
                    loss = cfg["vf_coef"] * v_loss
                else:
                    loss = actor_loss + cfg["vf_coef"] * v_loss - ent_coef * ent
                if anchor_coef > 0.0:
                    with torch.no_grad():
                        af_logits, am_logits, _ = anchor_model(
                            bsp[idx].view(mb * U, N_SPATIAL, BOARD, BOARD).to(device).float(),
                            bgl[idx].view(mb * U, N_GLOBAL).to(device).float())
                        af_logits = af_logits.view(mb, U, N_FARMER)
                        am_logits = am_logits.view(mb, U, N_MARKET)[:, 0]
                    afd = masked_categorical(af_logits, bfmk[idx].to(device))
                    amd = masked_categorical(am_logits, bmmk[idx].to(device))
                    kl_anchor = (torch.distributions.kl_divergence(fd, afd).sum(-1)
                                 + torch.distributions.kl_divergence(md, amd)).mean()
                    loss = loss + anchor_coef * kl_anchor
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), cfg["max_grad_norm"])
                opt.step()
                ep_ent += ent.item(); ep_vl += v_loss.item()
                ep_kl += (blgp[idx].to(device) - new_logp).mean().item(); nb += 1

        hist["entropy"].append(ep_ent / nb)
        hist["kl"].append(ep_kl / nb)
        hist["vloss"].append(ep_vl / nb)
        recent = vec.final_moneys[-2 * N:] or [cfg["starting_money"]]
        hist["final_money"].append(float(np.mean(recent)))
        opp_summary = league.summary()
        hist["vs_opponent"].append(opp_summary)

        done_epochs = epoch + 1
        if league.should_snapshot(done_epochs):
            tag = f"snap_ep{done_epochs}"
            league.add(model.state_dict(), tag)
            print(f"  -> league snapshot: {tag} (pool {len(league.snapshots)})")

        greedy_str = ""
        if greedy_fn is not None and done_epochs % cfg["greedy_check_every"] == 0:
            g_score = float(np.mean(greedy_fn(model)))
            hist["greedy"].append((done_epochs, g_score))
            greedy_str = f" | GREEDY ${g_score:.0f}"

        if epoch % cfg["log_every"] == 0:
            vs = " | ".join(f"{t}:{m:.0f}(n={n})" for t, (n, m) in opp_summary.items())
            print(f"ep {epoch:03d} | final$ {np.mean(recent):8.0f} | ent {ep_ent/nb:5.2f} "
                  f"| kl {ep_kl/nb:+.4f} | vloss {ep_vl/nb:7.3f} | ec {ent_coef:.4f}"
                  f"{' | WARM' if warming else ''}"
                  f"{f' | anch {anchor_coef:.3f}' if anchor_coef > 0 else ''}"
                  f" | {time.time()-t0:.0f}s"
                  f" | vs[{vs}] | auto_sell={cs.auto_sell}{greedy_str}")
        if (epoch + 1) % cfg["ckpt_every"] == 0:
            p = os.path.join(CKPT_DIR, f"ppo_ep{epoch+1}.pt")
            torch.save(model.state_dict(), p)
            print("  -> checkpoint", p)
    return hist

### ▶️ Run PPO

SMOKE: 3 tiny epochs — checking plumbing and finite losses. Real run: `final$` should start near the BC level (~$3,800) and climb as the policy learns what the expert never does: multiple tiles, better crops, land, animals, hands, sell timing. Watch `vs[snapshot_ep…]` to see it play against its own past.

In [ ]:
curriculum = build_curriculum(CONFIG)
history = train_ppo(model, vec, CONFIG, league, curriculum,
                    anchor_model=anchor_model,
                    greedy_fn=lambda m: evaluate(m, "starter", n_games=1, verbose=False))
final_path = os.path.join(CKPT_DIR, "ppo_final.pt")
torch.save(model.state_dict(), final_path)
print("saved", final_path)

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(history["final_money"])
    ax[0].axhline(3000, ls="--", c="gray", label="do-nothing")
    ax[0].axhline(3586, ls="--", c="green", label="starter expert")
    ax[0].set_title("Mean final money"); ax[0].legend()
    ax[1].plot(history["entropy"]); ax[1].set_title("Policy entropy")
    ax[2].plot(history["kl"]); ax[2].set_title("Approx KL / update")
    plt.tight_layout(); plt.show()
except ImportError:
    print("history:", {k: v for k, v in history.items() if k != "vs_opponent"})

## 10. Final evaluation

Greedy full seasons vs `starter`, `random`, and a frozen snapshot of the BC init (a fixed reference point — is PPO actually improving past the warm start?).

In [ ]:
if SMOKE:
    evaluate(model, "starter", n_games=1)
else:
    evaluate(model, "starter", n_games=10, render_first=True)
    evaluate(model, "random",  n_games=10)
    bc_ref = SnapshotOpponent(torch.load(os.path.join(CKPT_DIR, "bc_init.pt"),
                                         map_location="cpu"))
    evaluate(model, bc_ref, n_games=5)

## 11. Package the Kaggle submission

`submission.py` is assembled from the **live objects in this notebook** — constants via `repr()` of the runtime values, functions via `inspect.getsource` — so the deployed agent encodes/masks/decodes *exactly* like training, by construction.

Deployment selling strategy: the policy's learned SELL actions all season (`auto_sell=False`), with **forced liquidation on the last day** (`auto_sell=True` when `day >= 29`) so nothing valuable is left unsold when the whistle blows. The last `wheat_reserve` wheat are never sold (feed stock).

Upload with:
```bash
kaggle competitions submit kaggriculture -f submission.tar.gz -m "PPO v2.1"
```

In [ ]:
def write_submission(out_py):
    const_names = ["CROPS", "PRODUCTS", "ANIMALS", "SEED_COST", "CROP_FIRST_YIELD",
                   "ANIMAL_COST", "ANIMAL_STRUCTURE", "ANIMAL_PRODUCT", "LAND_PRICES",
                   "BOARD", "SHED_ADJACENT", "SHED_CAPACITY", "MAX_MARKET_ORDERS",
                   "MAX_HANDS", "WHEAT_PICKUP_N", "FARM_HAND_COST_MULT", "SELLABLE",
                   "MARKET_I0", "PRICE_FLOOR", "HINGE_GAIN", "MARKET_PARAMS",
                   "FARMER_ACTIONS", "MARKET_ACTIONS", "N_FARMER", "N_MARKET",
                   "N_SPATIAL", "N_GLOBAL"]
    g = globals()
    const_block = "\n".join(f"{n} = {g[n]!r}" for n in const_names)
    func_block = "\n\n".join(inspect.getsource(f) for f in
                             [_shape, market_price, _fib, hire_cost, _unit_pos,
                              encode_obs2, action_masks2, decode_farmer_action,
                              decode_market_action, build_action_dict2])
    try:
        model_src = inspect.getsource(FarmPolicy)
    except (OSError, TypeError):
        # some kernels can't source-extract classes; embed the canonical class
        model_src = """class FarmPolicy(nn.Module):
    def __init__(self, n_spatial=N_SPATIAL, n_global=N_GLOBAL,
                 n_farmer=N_FARMER, n_market=N_MARKET):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(n_spatial, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.mlp = nn.Sequential(nn.Linear(n_global, 64), nn.ReLU())
        torso_in = 64 * BOARD * BOARD + 64
        self.torso = nn.Sequential(nn.Linear(torso_in, 512), nn.ReLU())
        self.farmer_head = nn.Linear(512, n_farmer)
        self.market_head = nn.Linear(512, n_market)
        self.value_head  = nn.Linear(512, 1)

    def forward(self, spatial, glob):
        z = torch.cat([self.conv(spatial), self.mlp(glob)], dim=1)
        z = self.torso(z)
        return self.farmer_head(z), self.market_head(z), self.value_head(z).squeeze(-1)"""

    template = """
import os
import math
import numpy as np
import torch
import torch.nn as nn

__CONST_BLOCK__

__FUNC_BLOCK__

__MODEL_SRC__

WHEAT_RESERVE = __WHEAT_RESERVE__
LIQUIDATION_DAY = 29

_model = None

def _load():
    global _model
    if _model is None:
        _model = FarmPolicy()
        candidates = ["model.pth", "/kaggle_simulations/agent/model.pth"]
        try:  # __file__ is undefined when kaggle-environments execs this source
            candidates.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "model.pth"))
        except NameError:
            pass
        for cand in candidates:
            if os.path.exists(cand):
                _model.load_state_dict(torch.load(cand, map_location="cpu"))
                break
        else:
            print("WARNING: model.pth not found; running an untrained net")
        _model.eval()
    return _model

def agent(obs, config):
    model = _load()
    auto_sell = obs["day"] >= LIQUIDATION_DAY      # learned timing, forced liquidation at the end
    farm = obs["farms"][obs["player"]]
    n_hands = len(farm["hands"])
    S, G, FM = [], [], []
    for u in range(1 + n_hands):
        s, g = encode_obs2(obs, unit=u)
        fm, mm = action_masks2(obs, unit=u, auto_sell=auto_sell, wheat_reserve=WHEAT_RESERVE)
        S.append(s); G.append(g); FM.append(fm)
    with torch.no_grad():
        f_logits, m_logits, _ = model(torch.tensor(np.stack(S)), torch.tensor(np.stack(G)))
        f_logits = f_logits.masked_fill(~torch.tensor(np.stack(FM)), -1e9)
        m_logits = m_logits[0].masked_fill(~torch.tensor(mm), -1e9)
        fa = f_logits.argmax(-1).tolist()
        ma = int(m_logits.argmax(-1))
    return build_action_dict2(fa[0], ma, fa[1:], obs, auto_sell=auto_sell,
                              wheat_reserve=WHEAT_RESERVE)
"""
    code_text = (template
                 .replace("__CONST_BLOCK__", const_block)
                 .replace("__FUNC_BLOCK__", func_block)
                 .replace("__MODEL_SRC__", model_src)
                 .replace("__WHEAT_RESERVE__", repr(CONFIG["wheat_reserve"])))
    with open(out_py, "w") as f:
        f.write(code_text)
    print("wrote", out_py)

submission_py = os.path.join(WORKDIR, "submission.py")
write_submission(submission_py)

import shutil, tarfile
model_pth = os.path.join(WORKDIR, "model.pth")
shutil.copyfile(final_path, model_pth)
tar_path = os.path.join(WORKDIR, "submission.tar.gz")
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(submission_py, arcname="submission.py")
    tar.add(model_pth, arcname="model.pth")
print("packaged", tar_path)

### ✅ Submission smoke test

Full 720-turn match with the packaged agent; asserts both players reach DONE. Also plays a snapshot-vs-starter game to re-verify the multi-unit path end to end.

In [ ]:
def smoke_test_submission():
    env = make("kaggriculture", configuration={"episodeSteps": 720, "seed": 999}, debug=True)
    env.run([os.path.join(WORKDIR, "submission.py"), "starter"])
    st = env.state
    for i in range(2):
        print(f"player {i}: status={st[i].status} reward={st[i].reward}")
    assert all(st[i].status == "DONE" for i in range(2)), "agent crashed or sent invalid actions!"
    print("submission agent is legal and completes a full match.")

smoke_test_submission()

## 12. Throughput: the multiprocessing runner (optional)

`mp_worker.py` (shipped next to this notebook) is a drop-in `VecKag21` replacement that farms envs out to worker processes — **3.2× throughput at 32 envs × 4 workers** on this machine (1,243 env-steps/s vs 391), and it wins already at 2 workers. It also runs frozen-snapshot opponents *inside* the workers (pool tag `"policy"`), which is what makes league self-play cheap.

Rules of use:
* BC collection and temperature calibration stay on the in-process `VecKag21` (workers don't expose `.envs`).
* Swap `MpVecKag` in for the PPO loop; map the curriculum to `set_pool()` / `set_sell_mode()` / `update_snapshot()`.
* PPO rollout storage grows with `n_envs` — at 32 envs × 5 units, fp16 obs storage is ~2.8 GB per epoch. Reduce `n_envs` or `ppo_epochs` if RAM is tight.

In [ ]:
import sys
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

try:
    import importlib
    import mp_worker
    importlib.reload(mp_worker)

    mp_vec = mp_worker.MpVecKag(n_envs=CONFIG["n_envs"], cfg=CONFIG,
                                opponent_pool=["starter", "random"],
                                n_workers=2, base_seed=0)
    S, G, FM, MM = mp_vec.reset()
    print("MpVecKag batch:", S.shape, G.shape, FM.shape, MM.shape)
    t0 = time.time()
    for _t in range(10):
        fa = torch.stack([torch.tensor([np.random.choice(np.where(m.numpy())[0]) for m in u]) for u in FM])
        ma = torch.stack([torch.tensor(np.random.choice(np.where(m.numpy())[0])) for m in MM])
        (S, G, FM, MM), r, d = mp_vec.step(fa, ma)
    dt = time.time() - t0
    print(f"{CONFIG['n_envs'] * 10 / dt:.0f} env-steps/s over 10 steps (incl. worker startup amortized)")
    mp_vec.close()
    print("MpVecKag OK ✓")
except Exception as e:
    print("MpVecKag unavailable (fine for local testing):", type(e).__name__, e)

### Using MpVecKag in the PPO loop

```python
# after BC + temperature calibration (which use the in-process vec):
mp_vec = mp_worker.MpVecKag(n_envs=32, cfg=CONFIG,
                            opponent_pool=["starter", "random"],
                            n_workers=4, snapshot=mp_worker.make_snapshot(model))
# in train_ppo, replace vec.set_sell_mode(...) with mp_vec.set_sell_mode(...),
# league snapshots with mp_vec.update_snapshot(mp_worker.make_snapshot(model)),
# and opponent mixing with mp_vec.set_pool([...weighted tags...]).
# vec.final_moneys and mp_vec.results give the same metrics.
```

## 13. What's deliberately left for v2.2+

1. **Richer expert for BC**: the starter bot never moves, hires, or touches animals — BC caps at single-tile carrots and PPO must discover everything else. A hand-written goose/melon expert (or a strong trained checkpoint via DAgger) would raise the floor.
2. **Credit assignment for the animal chain**: buy→pickup→place→feed-daily is a ~10-turn chain with nightly re-pickups. If PPO struggles, add a short animal-only curriculum phase (coop pre-built, goose pre-placed) rather than reward shaping.
3. **Shed-capacity management**: nothing stops the policy from overflowing the 100-item shed on big harvest days (overflow is discarded).
4. **Opponent-aware play**: the shared market means a snapshot that floods wheat depresses prices for both players; explicit price-response (sell when price > threshold) is the simplest version.
5. **Bigger model / longer training**: the torso is one 512-wide layer; the game supports much more. With `MpVecKag` the env is no longer the bottleneck.